# Building a PMO

In this tutorial we will go through the steps to build a PMO, utilising the functions within the pmo-tools package. 

* Experiment Info - DONE
* Specimen Info - DONE 
* Sequencing Info - DONE
* Panel Info - DONE
* Representative mhap sequences - DONE
* microhaplotypes detected - DONE 
* Demultiplexed experiment samples - DONE
* TarAmpBioinformaticsInfo - DONE

TODO: 
* come up with a good example
* Run the data through the pipeline
* Download metadata from SRA
* Meat out the documentation 

In [1]:
# !pip install pmo-tools 

In [1]:
import pandas as pd

In [2]:
from pmotools.json_convertors.microhaplotype_table_to_pmo_dict import microhaplotype_table_to_pmo_dict
from pmotools.json_convertors.metatable_to_json_meta import experiment_info_table_to_json, specimen_info_table_to_json
from pmotools.json_convertors.panel_information_to_pmo_dict import panel_info_table_to_pmo_dict
from pmotools.json_convertors.demultiplexed_targets_to_pmo_dict import demultiplexed_targets_to_pmo_dict

## Allele Table 

In [3]:
example_allele_table = pd.read_csv('../example_data/dada2.clusters.txt', sep='\t')
example_allele_table.head()

,sampleID,locus,asv,reads,allele,norm.reads.locus,n.alleles
0,sample1,Pf3D7_01_v3-145388-145662-1A,GATATGTTTAAATATATGATTCTCGAAAAAACTTTTTTTATTTTTT...,15,Pf3D7_01_v3-145388-145662-1A.1,1.0,1
1,sample1,Pf3D7_01_v3-162867-163115-1A,ATATACCAATAATACTTTTTTTTTTAAATAATGTAAAAAATGATTT...,86,Pf3D7_01_v3-162867-163115-1A.1,1.0,1
2,sample1,Pf3D7_01_v3-181512-181761-1A,TTCATTATTGTTTTCATTCTTTTTTTAACTAAAACTATTCATCTCA...,214,Pf3D7_01_v3-181512-181761-1A.1,1.0,1
3,sample1,Pf3D7_01_v3-194742-194973-1B,TACCTATAAAAATGAAAAAAATAAAGAAGATAAATATGGAAAAAAT...,343,Pf3D7_01_v3-194742-194973-1B.1,1.0,1
4,sample1,Pf3D7_01_v3-455794-456054-1A,AGAAAAAAAATTTATTAAGAGGTATTTCGATTTTAAAAATTTAAGA...,18,Pf3D7_01_v3-455794-456054-1A.1,1.0,1


In [4]:
microhaplotype_dict = microhaplotype_table_to_pmo_dict(example_allele_table, '2024-07-29')

## Panel Info 

Below we show how we had to convert the panel information from the madhatter pipeline into the correct format for PMO

In [5]:
madhatter_panel_info = pd.read_csv('../example_data/v4_amplicon_info.tsv', sep='\t')
madhatter_panel_info.head()

,amplicon,amplicon_start,amplicon_end,ampInsert_start,ampInsert_end,rev_primer,amplicon_length,ampInsert_length,fwd_primer
0,Pf3D7_01_v3-145388-145662-1A,145388,145662,145421,145630,AAAATGTCCAATATGTCAAGGTATATTAAAGT,274,209,CCTGAGTTTTAAGTGAATGAATATATTTTTGTT
1,Pf3D7_01_v3-162867-163115-1A,162867,163115,162889,163092,TGTGTGCTTTGTCGTTGATTCAT,248,203,TACTACCGATCATCAAGCCGAA
2,Pf3D7_01_v3-181512-181761-1A,181512,181761,181545,181729,TAGTTTAAATCTATACTTGTCTCACCTGAACA,249,184,CTTTTCATATTTGTCTATTAGCTTTTTCAAACC
3,Pf3D7_01_v3-455794-456054-1A,455794,456054,455827,456021,GTGTTTCATTATTTTAGACACATTCAGGAATTT,260,194,ACAATGTAGAACAATATATAAAACTGGAAAAGA
4,Pf3D7_01_v3-528859-529104-1A,528859,529104,528890,529073,AATCATTTTATCCCACTTATTTATCTCGTCT,245,183,CTTAGTTTAGATTTGCCTACAATATTTGCAC


In [6]:
target_genome_info = {
			"gff_url" : "https://plasmodb.org/common/downloads/release-65/Pfalciparum3D7/gff/data/PlasmoDB-65_Pfalciparum3D7.gff",
			"name" : "3D7",
			"taxon_id" : 5833,
			"url" : "https://plasmodb.org/common/downloads/release-65/Pfalciparum3D7/fasta/data/PlasmoDB-65_Pfalciparum3D7_Genome.fasta",
			"version" : "2020-09-01"
		},

In [7]:
#  A C T G A C T G A C  T  G  A  C  T  G  A  C  T  G
#  1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
# 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20

# if first 4 are primer 
# original cooridnates = 1-4
# new coordinate = 0-4

# amplicon_end-

In [8]:
# Add on chromosome column
madhatter_panel_info['chrom'] = [chr[0] for chr in madhatter_panel_info.amplicon.str.split('-')]

# Add columns for now with information on primer length
madhatter_panel_info['fwd_primer_len'] = [len(p) for p in madhatter_panel_info.fwd_primer]
madhatter_panel_info['rev_primer_len'] = [len(p) for p in madhatter_panel_info.rev_primer]

# Convert to 1 based coordinates for our primers  
madhatter_panel_info['fwd_primer_start_0_based'] = madhatter_panel_info.amplicon_start-1
madhatter_panel_info['fwd_primer_end_0_based'] = madhatter_panel_info.fwd_primer_start_0_based+madhatter_panel_info.fwd_primer_len
madhatter_panel_info['rev_primer_start_0_based'] = madhatter_panel_info.amplicon_end-madhatter_panel_info.rev_primer_len
madhatter_panel_info['rev_primer_end_0_based'] = madhatter_panel_info.amplicon_end

# We trim one base off of each end in the pipeline, create insert coordinate 
madhatter_panel_info['insert_start_0_based'] = madhatter_panel_info.fwd_primer_end_0_based+1
madhatter_panel_info['insert_end_0_based'] = madhatter_panel_info.rev_primer_start_0_based-1

In [9]:
# no strand added - 
madhatter_panel_info['strand'] = '+'

In [10]:
madhatter_panel_info['gene_id'] = "gene"

In [11]:
panel_information_dict = panel_info_table_to_pmo_dict(madhatter_panel_info, "mad4hatter_poolsD1R1R2", target_genome_info, target_id_col="amplicon", 
                             forward_primers_seq_col="fwd_primer",forward_primers_start_col="fwd_primer_start_0_based", 
                             forward_primers_end_col = "fwd_primer_end_0_based",reverse_primers_seq_col="rev_primer", 
                             reverse_primers_start_col="rev_primer_start_0_based", reverse_primers_end_col="rev_primer_end_0_based",
                             insert_start_col="insert_start_0_based",insert_end_col="insert_end_0_based",
                             )

In [12]:
# add on gene_id and strand 
# Write a function that takes table with everything in it and turns it into the json format 
# Add option to generate all of this from the primers 
# Add validation

# Sequencing info

In [13]:
sequencing_infos ={
		"Mozambique2018" : 
		{
			"lib_kit" : "TruSeq i5/i7 barcode primers",
			"lib_layout" : "paired-end",
			"lib_screen" : "40 µL reaction containing 10 µL of bead purified digested product, 18μL of nuclease-free water, 8μL of 5X secondary PCR master mix, and 5 µL of 10 µM TruSeq i5/i7 barcode primers",
			"nucl_acid_amp" : "https://www.paragongenomics.com/targeted-sequencing/amplicon-sequencing/cleanplex-ngs-amplicon-sequencing/",
			"nucl_acid_date" : "2019-07-15",
			"nucl_acid_ext" : "https://www.paragongenomics.com/targeted-sequencing/amplicon-sequencing/cleanplex-ngs-amplicon-sequencing/",
			"pcr_cond" : "10 min at 95°C, 13 cycles for high density samples (or 15 cycles for low density samples) of 15 sec at 98°C and 75 sec at 60°C",
			"seq_center" : "UCSF",
			"seq_date" : "2019-07-15",
			"seq_instrument" : "NextSeq 550 instrument",
			"sequencing_info_id" : "run1"
		}
	}

# Demultiplexed experiment samples

In [14]:
amplicon_coverage = pd.read_csv('../example_data/amplicon_coverage.txt', sep='\t')
amplicon_coverage.head()

,SampleID,Locus,Reads,OutputDada2,OutputPostprocessing
0,GM-1B-D4-100-P2-3_S22_L001,Pf3D7_01_v3-145388-145662-1A,0,0,0
1,GM-1B-D4-100-P2-3_S22_L001,Pf3D7_01_v3-162867-163115-1A,0,0,0
2,GM-1B-D4-100-P2-3_S22_L001,Pf3D7_01_v3-181512-181761-1A,0,0,0
3,GM-1B-D4-100-P2-3_S22_L001,Pf3D7_01_v3-455794-456054-1A,0,0,0
4,GM-1B-D4-100-P2-3_S22_L001,Pf3D7_01_v3-528859-529104-1A,0,0,0


In [15]:
demultiplexed_targets_pmo = demultiplexed_targets_to_pmo_dict(amplicon_coverage, sampleID_col = 'SampleID', target_id_col='Locus',read_count_col ='Reads')

# TarAmpBioinformaticsInfo

In [16]:
taramp_bioinformatics_infos = {
    "Mozambique2018-SeekDeep" : 
    {
        "demultiplexing_method" : 
        {
            "program" : "SeekDeep extractorPairedEnd",
            "purpose" : "Takes raw paired-end reads and demultiplexes on primers and does QC filtering",
            "version" : "v2.6.5"
        },
        "denoising_method" : 
        {
            "additional_argument" : "--illumina --qualThres 25,20 --trimFront 1 --trimBack 1",
            "program" : "SeekDeep qluster",
            "purpose" : "Takes sequences per sample per target and clusters them",
            "version" : "v2.6.5"
        },
        "population_clustering_method" : 
        {
            "additional_argument" : "--strictErrors  --illumina --removeOneSampOnlyOneOffHaps --excludeCommonlyLowFreqHaplotypes --excludeLowFreqOneOffs --rescueExcludedOneOffLowFreqHaplotypes",
            "program" : "SeekDeep processClusters",
            "purpose" : "Compare across samples for each target to create population level identifiers and do post artifact cleanup",
            "version" : "v2.6.5"
        },
        "tar_amp_bioinformatics_info_id" : "Mozambique2018-SeekDeep"
    }
}

# Metadata 

In [17]:
# "experiment_sample_id" : "8025874217",
# "panel_id" : "heomev1",
# "plate_col" : 12,
# "plate_name" : "8",
# "plate_row" : "C",
# "sequencing_info_id" : "Mozambique2018",
# "specimen_id" : "8025874217"

In [18]:
import itertools

# Create the list
letters = list(itertools.chain.from_iterable([char] * 12 for char in 'ABCDEFGH'))
cols = [i for i in range(1, 13)] * 8
letters = letters*4
cols = cols*4
plates = list(itertools.chain.from_iterable([char] * 96 for char in ['plate1','plate2','plate3','plate4']))

In [19]:
experiment_infos = amplicon_coverage[['SampleID']]
experiment_infos['SampleID'] = [f'sample{i}' for i in range(len(experiment_infos))]
experiment_infos['panel_id'] = 'Mad4hatter'
experiment_infos['plate_name'] = plates[:len(experiment_infos)]
experiment_infos['plate_row'] = letters[:len(experiment_infos)]
experiment_infos['plate_col'] = cols[:len(experiment_infos)]
experiment_infos['sequencing_info_id'] = 'run1'
experiment_infos['specimen_id'] = experiment_infos.SampleID
experiment_infos

/var/folders/05/xlr96nt10rngc086j8jv1g_c0000gp/T/ipykernel_16598/824991483.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  experiment_infos['SampleID'] = [f'sample{i}' for i in range(len(experiment_infos))]
/var/folders/05/xlr96nt10rngc086j8jv1g_c0000gp/T/ipykernel_16598/824991483.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  experiment_infos['panel_id'] = 'Mad4hatter'
/var/folders/05/xlr96nt10rngc086j8jv1g_c0000gp/T/ipykernel_16598/824991483.py:4: SettingWithCopyWarning: 
A value is trying to be 

,SampleID,panel_id,plate_name,plate_row,plate_col,sequencing_info_id,specimen_id
0,sample0,Mad4hatter,plate1,A,1,run1,sample0
1,sample1,Mad4hatter,plate1,A,2,run1,sample1
2,sample2,Mad4hatter,plate1,A,3,run1,sample2
3,sample3,Mad4hatter,plate1,A,4,run1,sample3
4,sample4,Mad4hatter,plate1,A,5,run1,sample4
...,...,...,...,...,...,...,...
271,sample271,Mad4hatter,plate3,G,8,run1,sample271
272,sample272,Mad4hatter,plate3,G,9,run1,sample272
273,sample273,Mad4hatter,plate3,G,10,run1,sample273
274,sample274,Mad4hatter,plate3,G,11,run1,sample274


In [20]:
experiment_infos.rename(columns={'SampleID':'experiment_sample_id'}, inplace=True)

In [21]:
experiment_info_json = experiment_info_table_to_json(experiment_infos)

In [22]:
metadata = experiment_infos[['specimen_id']]
metadata['collection_country'] = "Mocambique"
metadata['collection_date'] = "2018-06-07"
metadata['collector'] = "Greenhouse, Bryan"
metadata['geo_admin3'] = "Inhassoro"
metadata['host_taxon_id'] = 1758
metadata['lat_lon'] = "-21.5535,35.1819"
metadata['parasite_density'] = 477719.34375
metadata['project_name'] = "MOZ2018"
metadata['samp_collect_device'] = "dried blood spot"
metadata['samp_store_loc'] = "UCSF Greenhouse Lab"
metadata['samp_taxon_id'] = 5833
# metadata['specimen_id'] = "8025874217"

/var/folders/05/xlr96nt10rngc086j8jv1g_c0000gp/T/ipykernel_16598/4245134413.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['collection_country'] = "Mocambique"
/var/folders/05/xlr96nt10rngc086j8jv1g_c0000gp/T/ipykernel_16598/4245134413.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['collection_date'] = "2018-06-07"
/var/folders/05/xlr96nt10rngc086j8jv1g_c0000gp/T/ipykernel_16598/4245134413.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

In [23]:
metadata.head()

,specimen_id,collection_country,collection_date,collector,geo_admin3,host_taxon_id,lat_lon,parasite_density,project_name,samp_collect_device,samp_store_loc,samp_taxon_id
0,sample0,Mocambique,2018-06-07,"Greenhouse, Bryan",Inhassoro,1758,"-21.5535,35.1819",477719.34375,MOZ2018,dried blood spot,UCSF Greenhouse Lab,5833
1,sample1,Mocambique,2018-06-07,"Greenhouse, Bryan",Inhassoro,1758,"-21.5535,35.1819",477719.34375,MOZ2018,dried blood spot,UCSF Greenhouse Lab,5833
2,sample2,Mocambique,2018-06-07,"Greenhouse, Bryan",Inhassoro,1758,"-21.5535,35.1819",477719.34375,MOZ2018,dried blood spot,UCSF Greenhouse Lab,5833
3,sample3,Mocambique,2018-06-07,"Greenhouse, Bryan",Inhassoro,1758,"-21.5535,35.1819",477719.34375,MOZ2018,dried blood spot,UCSF Greenhouse Lab,5833
4,sample4,Mocambique,2018-06-07,"Greenhouse, Bryan",Inhassoro,1758,"-21.5535,35.1819",477719.34375,MOZ2018,dried blood spot,UCSF Greenhouse Lab,5833


In [24]:
specimen_info_table_to_json(metadata)

{'sample0': {'specimen_id': 'sample0',
  'collection_country': 'Mocambique',
  'collection_date': '2018-06-07',
  'collector': 'Greenhouse, Bryan',
  'geo_admin3': 'Inhassoro',
  'host_taxon_id': 1758,
  'lat_lon': '-21.5535,35.1819',
  'parasite_density': 477719.34375,
  'project_name': 'MOZ2018',
  'samp_collect_device': 'dried blood spot',
  'samp_store_loc': 'UCSF Greenhouse Lab',
  'samp_taxon_id': 5833},
 'sample1': {'specimen_id': 'sample1',
  'collection_country': 'Mocambique',
  'collection_date': '2018-06-07',
  'collector': 'Greenhouse, Bryan',
  'geo_admin3': 'Inhassoro',
  'host_taxon_id': 1758,
  'lat_lon': '-21.5535,35.1819',
  'parasite_density': 477719.34375,
  'project_name': 'MOZ2018',
  'samp_collect_device': 'dried blood spot',
  'samp_store_loc': 'UCSF Greenhouse Lab',
  'samp_taxon_id': 5833},
 'sample2': {'specimen_id': 'sample2',
  'collection_country': 'Mocambique',
  'collection_date': '2018-06-07',
  'collector': 'Greenhouse, Bryan',
  'geo_admin3': 'Inhasso

In [11]:
from Bio import Entrez
import json 

# Set your email to use NCBI Entrez
Entrez.email = "kathryn_murie@hotmail.co.uk"


handle = Entrez.esearch(db="sra", term='PRJNA1040019')
record = Entrez.read(handle)
handle.close()

# Get a list of SRA IDs (SRA runs)
sra_ids = record["IdList"]

In [13]:
!esearch -db genome -query "22954[uid]" | \
elink -target bioproject | \
efetch -format xml | \
xtract -pattern DocumentSummary -element Salinity OxygenReq OptimumTemperature TemperatureRange Habitat

zsh:1: command not found: elink
zsh:1: command not found: xtract
zsh:1: command not found: efetch
zsh:1: command not found: esearch


In [12]:
#JSON formatted output
def get_assembly_summary_json(id):
    handle = Entrez.esummary(db="sra",id=id,report="full")
    record = Entrez.read(handle)
    #Convert raw output to json
    return(json.dumps(record, sort_keys=True,indent=4, separators=(',', ': ')))

#Test
for id in sra_ids:
    #print(get_raw_assembly_summary(id)) #For raw output
    print(get_assembly_summary_json(id)) #JSON Formatted

[
    {
        "CreateDate": "2023/11/14",
        "ExpXml": "<Summary><Title>Amplicon-seqeuncing of plasmodium falciparum form pregnant women and childrne in Mozambique</Title><Platform instrument_model=\"Illumina MiSeq\">ILLUMINA</Platform><Statistics total_runs=\"1\" total_spots=\"937151\" total_bases=\"294265414\" total_size=\"97719571\" load_done=\"true\" cluster_name=\"public\"/></Summary><Submitter acc=\"SRA1750351\" center_name=\"Barcelona Institute for Global Health\" contact_name=\"Nanna Brokhattingen\" lab_name=\"Malaria\"/><Experiment acc=\"SRX22516351\" ver=\"1\" status=\"public\" name=\"Amplicon-seqeuncing of plasmodium falciparum form pregnant women and childrne in Mozambique\"/><Study acc=\"SRP471673\" name=\"Plasmodium falciparum genetic diversity and drug resistance markers in southern Mozambique\"/><Organism taxid=\"5833\" ScientificName=\"Plasmodium falciparum\"/><Sample acc=\"SRS19528143\" name=\"\"/><Instrument ILLUMINA=\"Illumina MiSeq\"/><Library_descriptor><LI

In [48]:
# Fetch details for each SRA ID
handle = Entrez.esummary(db="sra", id=",".join(sra_ids))
summaries = Entrez.read(handle)
handle.close()

In [68]:
document_summary

'<Summary><Title>Amplicon-seqeuncing of plasmodium falciparum form pregnant women and childrne in Mozambique</Title><Platform instrument_model="Illumina MiSeq">ILLUMINA</Platform><Statistics total_runs="1" total_spots="673053" total_bases="197877582" total_size="64844371" load_done="true" cluster_name="public"/></Summary><Submitter acc="SRA1750351" center_name="Barcelona Institute for Global Health" contact_name="Nanna Brokhattingen" lab_name="Malaria"/><Experiment acc="SRX22516332" ver="1" status="public" name="Amplicon-seqeuncing of plasmodium falciparum form pregnant women and childrne in Mozambique"/><Study acc="SRP471673" name="Plasmodium falciparum genetic diversity and drug resistance markers in southern Mozambique"/><Organism taxid="5833" ScientificName="Plasmodium falciparum"/><Sample acc="SRS19528128" name=""/><Instrument ILLUMINA="Illumina MiSeq"/><Library_descriptor><LIBRARY_NAME>GenMoz_4051120248_RetroGM_PV4C1Pools1AB_S1040</LIBRARY_NAME><LIBRARY_STRATEGY>AMPLICON</LIBRARY

In [ ]:
<Summary><Title>Amplicon-seqeuncing of plasmodium falciparum form pregnant women and childrne in Mozambique</Title><Platform instrument_model="Illumina MiSeq">ILLUMINA</Platform><Statistics total_runs="1" total_spots="673053" total_bases="197877582" total_size="64844371" load_done="true" cluster_name="public"/></Summary><Submitter acc="SRA1750351" center_name="Barcelona Institute for Global Health" contact_name="Nanna Brokhattingen" lab_name="Malaria"/><Experiment acc="SRX22516332" ver="1" status="public" name="Amplicon-seqeuncing of plasmodium falciparum form pregnant women and childrne in Mozambique"/><Study acc="SRP471673" name="Plasmodium falciparum genetic diversity and drug resistance markers in southern Mozambique"/><Organism taxid="5833" ScientificName="Plasmodium falciparum"/><Sample acc="SRS19528128" name=""/><Instrument ILLUMINA="Illumina MiSeq"/><Library_descriptor><LIBRARY_NAME>GenMoz_4051120248_RetroGM_PV4C1Pools1AB_S1040</LIBRARY_NAME><LIBRARY_STRATEGY>AMPLICON</LIBRARY_STRATEGY><LIBRARY_SOURCE>GENOMIC</LIBRARY_SOURCE><LIBRARY_SELECTION>PCR</LIBRARY_SELECTION><LIBRARY_LAYOUT> <PAIRED/> </LIBRARY_LAYOUT></Library_descriptor><Bioproject>PRJNA1040019</Bioproject><Biosample>SAMN38241489</Biosample>

In [3]:
import pysradb

/Users/kmurie/Documents/environments/pmotools/lib/python3.11/site-packages/pysradb/utils.py:14: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [15]:
pysradb metadata

SyntaxError: invalid syntax (2911063153.py, line 1)

In [17]:
pysradb -help

TypeError: unsupported operand type(s) for -: 'module' and '_Helper'

In [20]:
handle = Entrez.esearch(db="sra", term='PRJNA1040019')
record = Entrez.read(handle)
handle.close()

HTTPError: HTTP Error 400: Bad Request

In [19]:
from Bio import Entrez
import json

#Increase query limit to 10/s & get warnings
# Entrez.email = ""
#Get one from https://www.ncbi.nlm.nih.gov/account/settings/ page
# Entrez.api_key=""

term="PRJNA1040019"
#Finds the ids associated with the assembly
def get_ids(term):
    ids = []
    handle = Entrez.esearch(db="assembly", term=term)
    record = Entrez.read(handle)
    ids.append(record["IdList"])
    return ids

#Fetch raw output
def get_raw_assembly_summary(id):
    handle = Entrez.esummary(db="assembly",id=id,report="full")
    record = Entrez.read(handle)
    #Return individual fields
    #XML output: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=assembly&id=79781&report=%22full%22
    #return(record['DocumentSummarySet']['DocumentSummary'][0]['AssemblyName']) #This will return the Assembly name
    return(record)

#JSON formatted output
def get_assembly_summary_json(id):
    handle = Entrez.esummary(db="assembly",id=id,report="full")
    record = Entrez.read(handle)
    #Convert raw output to json
    return(json.dumps(record, sort_keys=True,indent=4, separators=(',', ': ')))

#Test
for id in get_ids(term):
    #print(get_raw_assembly_summary(id)) #For raw output
    print(get_assembly_summary_json(id)) #JSON Formatted

HTTPError: HTTP Error 400: Bad Request

In [29]:
df = pd.read_csv('/Users/kmurie/Downloads/filereport_read_run_PRJNA1040019_tsv (1).txt', sep='\t')

In [35]:
df.run_accession.nunique()

558

In [43]:
acc = [i[1] for i in df.experiment_alias.str.split('_')]
len(np.unique(acc))

558

In [41]:
import numpy as np